In [24]:
from dotenv import load_dotenv
# Load environment variables from .env file

import os
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [15]:
from langchain_openai import ChatOpenAI
from langchain_classic.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType

# --- Initialize the LLM ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_API_KEY)

# --- Define your tools ---
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b

def get_weather(city: str) -> str:
    """Pretend to get the weather (demo mode)."""
    return f"The weather in {city} is sunny ☀️ (demo mode)."

# Wrap tools for LangChain
tools = [
    Tool(
        name="Add Numbers",
        func=lambda x: add_numbers(*[float(i) for i in x.split()]),
        description="Add two numbers provided as input, separated by a space. Example: '2.5 7.3'"
    ),
    Tool(
        name="Get Weather",
        func=get_weather,
        description="Get the weather for a given city name."
    )
]


# --- Initialize the agent ---
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)


In [16]:
query = "Hi! Can you tell me the weather in Paris and add 2.5 + 7.3?"
response = agent.invoke(query)
print("\n💬 Agent Answer:\n", response["output"])




> Entering new AgentExecutor chain...
I need to get the weather for Paris and also perform the addition of the numbers 2.5 and 7.3. I'll start with the weather.

Action: Get Weather  
Action Input: "Paris"  
Observation: The weather in Paris is sunny ☀️ (demo mode).
Thought:I have the weather information for Paris. Now, I will proceed to add the numbers 2.5 and 7.3.

Action: Add Numbers  
Action Input: "2.5 7.3"  
Observation: 9.8
Thought:I now know the final answer, which includes both the weather in Paris and the result of the addition. 

Final Answer: The weather in Paris is sunny ☀️ and 2.5 + 7.3 equals 9.8.

> Finished chain.

💬 Agent Answer:
 The weather in Paris is sunny ☀️ and 2.5 + 7.3 equals 9.8.


In [22]:
llm.invoke("Who is Ada Lovelace?")

AIMessage(content="Ada Lovelace, born Augusta Ada Byron on December 10, 1815, is often regarded as one of the first computer programmers. She was the daughter of the famous poet Lord Byron and mathematician Annabella Milbanke. Lovelace is best known for her work on Charles Babbage's early mechanical general-purpose computer, the Analytical Engine.\n\nIn 1843, she translated an article written by the Italian mathematician Luigi Federico Federico, adding her own notes and expanding on the concepts. Her notes included what is considered the first algorithm intended to be processed by a machine, specifically for the Analytical Engine. This work laid the groundwork for future developments in computer science.\n\nLovelace's vision extended beyond mere calculations; she speculated on the potential of computers to manipulate symbols and create music and art, which was quite forward-thinking for her time. Despite her contributions being largely unrecognized during her lifetime, she has since be

In [17]:
import requests

def simple_duckduckgo_search(query: str) -> str:
    """Stable fallback web search using DuckDuckGo Instant Answer API."""
    try:
        resp = requests.get(
            "https://api.duckduckgo.com/",
            params={"q": query, "format": "json", "no_html": 1, "no_redirect": 1},
            timeout=10
        )
        data = resp.json()
        if data.get("AbstractText"):
            return data["AbstractText"]
        elif data.get("RelatedTopics"):
            results = [t.get("Text") for t in data["RelatedTopics"] if "Text" in t]
            return "\n".join(results[:3])
        else:
            return "No relevant results found."
    except Exception as e:
        return f"Search error: {e}"

tools = [
    Tool(
        name="DuckDuckGo Search",
        func=simple_duckduckgo_search,
        description="Look up people or topics on the web."
    )
]

# --- create agent with tool ---
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# --- ask the question ---
query = "Who is Ada Lovelace?"
response = agent.invoke(query)

print("\n💬 Agent Answer:\n", response["output"])




> Entering new AgentExecutor chain...
I need to gather information about Ada Lovelace, her contributions, and significance in history. 
Action: DuckDuckGo Search
Action Input: "Who is Ada Lovelace?"
Observation: Augusta Ada King, Countess of Lovelace, also known as Ada Lovelace, was an English mathematician and writer chiefly known for work on Charles Babbage's proposed mechanical general-purpose computer, the analytical engine. She was the first to recognise the machine had applications beyond pure calculation. Lovelace is often considered the first computer programmer. Lovelace was the only legitimate child of poet Lord Byron and reformer Anne Isabella Milbanke. Lord Byron separated from his wife a month after Ada was born, and died when she was eight. Although often ill in childhood, Lovelace pursued her studies assiduously. She married William King in 1835. King was a Baron, and was created Viscount Ockham and 1st Earl of Lovelace in 1838. The name Lovelace was chosen because Ada

In [23]:
agent.invoke("Who Kamila Kare?")



> Entering new AgentExecutor chain...
I need to find information about Kamila Kare to answer the question. 
Action: DuckDuckGo Search
Action Input: "Kamila Kare"
Observation: No relevant results found.
Thought:It seems that there are no significant results for "Kamila Kare." I may need to refine my search or consider that this person may not be widely known or documented online. 

Action: DuckDuckGo Search
Action Input: "Kamila Kare biography" 
Observation: No relevant results found.
Thought:It appears that there is still no information available about Kamila Kare. This could indicate that she is not a public figure or that her presence online is minimal. 

Action: DuckDuckGo Search
Action Input: "Kamila Kare news" 
Observation: No relevant results found.
Thought:It seems that there is a consistent lack of information regarding Kamila Kare across various search queries. This suggests that she may not be a widely recognized individual or that there is limited public information availa

{'input': 'Who Kamila Kare?',
 'output': 'There is no available information on Kamila Kare. She may not be a public figure or widely documented online.'}

In [25]:
import os
from tavily import TavilyClient

# Make sure this exists in your .env

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def simple_tavily_search(query: str) -> str:
    """Web search using Tavily API."""
    try:
        res = tavily_client.search(
            query=query,
            search_depth="basic",   # or "advanced"
            max_results=5
        )

        results = res.get("results", [])
        if not results:
            return "No relevant results found."

        lines = []
        for r in results[:3]:
            title = r.get("title", "No title")
            content = r.get("content", "")
            url = r.get("url", "")
            lines.append(f"- {title}\n  {content}\n  {url}")
        return "\n".join(lines)

    except Exception as e:
        return f"Search error: {e}"

tools = [
    Tool(
        name="Tavily Search",
        func=simple_tavily_search,
        description="Search the web for up-to-date information."
    )
]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)



In [26]:
query = "Who is Kamila Kare?"
response = agent.invoke({"input": query})  # recommended shape
print("\n💬 Agent Answer:\n", response["output"])



> Entering new AgentExecutor chain...
I need to find current information about Kamila Kare to provide an accurate answer. 
Action: Tavily Search
Action Input: "Kamila Kare"
Observation: - Kamila Kare, PhD - Sanofi - LinkedIn
  Kamila Kare, PhD
AI Applied Scientist
Montigny-le-Bretonneux, Île-de-France, France
500 connections, 1643 followers


About
I am a Senior Data Scientist. I’m passionate about developing novel machine learning…


Experience
N/A


Education
Université Paris 1 Panthéon-Sorbonne
N/A
2018 - 2021-07
None
  https://fr.linkedin.com/in/kamila-kare-phd-572a87112
- ‪kamila kare‬ - ‪Google 학술 검색‬
  kamila kare. Université Paris 1, Panthéon Sorbonne. univ-paris1.fr의 이메일 확인됨. Statistique Mathématique. 학술자료인용공개 액세스공동 저자. 제목.
  https://scholar.google.com.co/citations?user=ulEGaisAAAAJ&hl=ko
- Kamila Kare - Facebook
  Kamila Kare. Photos. See all. Others With a Similar Name. See all · Kamilla Karé · Milla Santos · Yeismar Kamila Rodriguez Moreno · Kamila Gustavo.
  https://www.